微调

In [ ]:
pip install matplotlib==3.8.0

In [ ]:
pip install transformers==4.49.0

In [ ]:
pip install scikit-learn==1.4.2

In [ ]:
pip install numpy==1.26.4

In [ ]:
pip install pandas==2.1.4

In [ ]:
pip install accelerate

In [ ]:
pip uninstall torch torchvision torchaudio

In [ ]:
import torch
print(torch.__version__)
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support,roc_auc_score
import pandas as pd
import numpy as np

In [ ]:
if torch.cuda.is_available():    # 指定使用 cuda:1    
    device = torch.device("cuda")
else:   
    device = torch.device("cpu")

In [ ]:
# 本地数据地址
train_data_path = r"C:\Users\12225\Desktop\精神疾病数据库\depresson_text\new_data\train_before.csv"
dev_data_path = r"C:\Users\12225\Desktop\精神疾病数据库\depresson_text\new_data\dev_before.csv"
test_data_path = r"C:\Users\12225\Desktop\精神疾病数据库\depresson_text\new_data\test_before.csv"

# 读取数据
train_df = pd.read_csv(train_data_path)
train_df = train_df.dropna(subset=['patient_text'])
dev_df = pd.read_csv(dev_data_path)
dev_df = dev_df.dropna(subset=['patient_text'])
test_df = pd.read_csv(test_data_path)
test_df = test_df.dropna(subset=['patient_text'])

# 读取标签
real_labels = ['not depression', 'depression']

In [ ]:
# 按类别抽样（假设标签列名为'label'，请根据实际列名调整）
class_0_samples = train_df[train_df['depressed-before'] == 0].sample(n=50, random_state=42)
class_1_samples = train_df[train_df['depressed-before'] == 1].sample(n=50, random_state=42)

# 合并并再次打乱确保随机性
train_df = pd.concat([class_0_samples, class_1_samples], axis=0)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 验证结果
print("新训练集类别分布：")
print(train_df['depressed-before'].value_counts())

In [ ]:
# 按类别抽样（假设标签列名为'label'，请根据实际列名调整）
class_0_samples = dev_df[dev_df['depressed-before'] == 0].sample(n=50, random_state=42)
class_1_samples = dev_df[dev_df['depressed-before'] == 1].sample(n=50, random_state=42)

# 合并并再次打乱确保随机性
dev_df = pd.concat([class_0_samples, class_1_samples], axis=0)
dev_df = dev_df.sample(frac=1, random_state=42).reset_index(drop=True)

# 验证结果
print("新验证集类别分布：")
print(dev_df['depressed-before'].value_counts())

In [ ]:
train_df

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_id = "modernbertlarge/"
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
from torch.utils.data import Dataset
import torch

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        # 使用 tokenizer 对文本进行编码，并进行截断和填充
        self.encodings = tokenizer(texts,  max_length = 1280,truncation=True, padding='max_length', return_tensors="pt")
        # 存储标签数据
        self.labels = labels

    def __getitem__(self, idx):
        # 将编码后的文本数据转换为 PyTorch 张量
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()} #item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # 添加标签数据
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)
        
# 创建类别到索引的映射
label_to_index = {label: idx for idx, label in enumerate(train_df['depressed-before'].unique())}

# 使用映射来更新标签
train_labels = [label_to_index[label] for label in train_df['depressed-before']]
dev_labels = [label_to_index[label] for label in dev_df['depressed-before']]

# 然后在创建数据集时使用这些更新后的标签
train_dataset = TextDataset(train_df['patient_text'].tolist(), train_labels)
dev_dataset = TextDataset(dev_df['patient_text'].tolist(), dev_labels)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from transformers import TrainerCallback, TrainingArguments, TrainerState, TrainerControl
import os
os.environ["TORCH_COMPILE_DISABLE"] = "1"
label2id, id2label = dict(), dict()
for i, label in enumerate(real_labels):
    label2id[label] = str(i)
    id2label[str(i)] = label
model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=len(train_df['depressed-before'].unique()),label2id=label2id, id2label=id2label).to(device) 

# 定义训练参数
training_args = TrainingArguments(
    output_dir='./result9-10',  # 训练结果和模型的输出目录
    num_train_epochs=5, # 训练的轮数
    per_device_train_batch_size=1,  # 每个设备的训练批次大小
    per_device_eval_batch_size=1,  # 每个设备的评估批次大小
    warmup_steps=50,  # 学习率预热步数
    learning_rate=1e-6,
    optim="adamw_torch_fused",
    weight_decay=0.1,  # 权重衰减（L2正则化）
    warmup_ratio=0.1,
    bf16=True,
    logging_dir='./logs',  # 日志文件的存储目录
    logging_steps=100,  # 记录日志的步数间隔
    eval_strategy="epoch",  # 评估策略，每个epoch结束时进行评估
    save_strategy="epoch",  # 模型保存策略，每个epoch结束时保存模型
    save_total_limit=2,
    load_best_model_at_end=True,  # 训练结束后加载最佳模型
    push_to_hub=False,  # 是否将模型推送到Hugging Face Hub
    gradient_accumulation_steps=2,
    max_grad_norm=1.0,  # 梯度裁剪阈值
    disable_tqdm=False
)

def compute_metrics(pred):  # 定义评估指标计算函数
    labels = pred.label_ids  # 真实标签
    preds = pred.predictions.argmax(-1)  # 预测标签
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')  # 计算精确率、召回率和F1分数
    acc = accuracy_score(labels, preds)  # 计算准确率
    auc = roc_auc_score(labels, preds)
    return {
        'accuracy': acc,  # 返回准确率
        'f1': f1,  # 返回F1分数
        'precision': precision,  # 返回精确率
        'recall': recall,  # 返回召回率
        'auc': auc  # 返回AUC分数
    }


class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, early_stopping_patience=3):
        self.early_stopping_patience = early_stopping_patience
        self.best_metric = None
        self.patience_counter = 0

    def on_evaluate(self, args: TrainingArguments, state: TrainerState, control: TrainerControl, metrics, **kwargs):
        # 假设监控验证集的loss（可根据需求改为其他指标如eval_accuracy）
        current_metric = metrics.get("eval_loss", None)
        
        if current_metric is None:
            return

        if self.best_metric is None:
            self.best_metric = current_metric
        elif current_metric < self.best_metric:  # 如果监控loss，指标越小越好
            self.best_metric = current_metric
            self.patience_counter = 0  # 重置计数器
        else:
            self.patience_counter += 1
            print(f"早停计数器: {self.patience_counter}/{self.early_stopping_patience}")

            if self.patience_counter >= self.early_stopping_patience:
                print("触发早停！")
                control.should_training_stop = True  # 停止训练
                control.should_save = True  # 确保保存最佳模型

# 使用方式
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics,
    #callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  # 添加回调
)
# trainer = Trainer(
#     model=model,  # 要训练的模型
#     args=training_args,  # 训练参数
#     train_dataset=train_dataset,  # 训练数据集
#     eval_dataset=dev_dataset,  # 评估数据集
#     compute_metrics=compute_metrics  # 评估指标计算函数
# )


trainer.train()  # 开始训练模型

trainer.evaluate()  # 在评估数据集上评估模型性能

In [ ]:
import triton
print(triton.__version__)

In [ ]:
import matplotlib.pyplot as plt
# 提取微调训练日志
fine_tune_logs = trainer.state.log_history
fine_tune_train_loss = [log["loss"] for log in fine_tune_logs if "loss" in log]
fine_tune_val_loss = [log["eval_loss"] for log in fine_tune_logs if "eval_loss" in log]
fine_tune_epochs = range(1, len(fine_tune_val_loss)+ 1)
#绘制微调训练和验证损失图像
plt.figure(figsize=(10,6))
plt.plot(range(1, len(fine_tune_train_loss) + 1), fine_tune_train_loss, label="Train Loss", marker="o")
plt.plot(fine_tune_epochs, fine_tune_val_loss, label="Validation Loss", marker="x")
plt.xlabel("steps")
plt.ylabel("Loss")
plt.title("Fine tuning TrainingI and Validation Loss")
plt.legend()
plt.grid()
plt.show()

In [ ]:
# 保存模型到本地/远程
output_dir = "/workspace/wsqstar/depberta-domain-classifier"
trainer.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)